# Netflix Content Strategy Analysis (2016-2021)

## Thesis
How did Netflix's content strategy evolve from 2016 to 2021,
and did COVID-19 change what they produced?

## Dataset
- Source: Kaggle Netflix Movies and TV Shows dataset
- 8,807 titles spanning 2008-2021
- Note: 9% of titles are missing country data and were excluded from country analysis

In [45]:
import pandas as pd 
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
import numpy as np

netflix_data = pd.read_csv('netflix_titles.csv')

## Section 1 - Setting the scene
A first look at the data: total titles, content type, and data quality notes.

In [46]:
netflix_data.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [47]:
netflix_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [48]:
netflix_data.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [49]:
# dropping missing values for descriptive variables: director, cast, and country
# dropping missing values since we will explore this in this analysis
netflix_data['director'] = netflix_data['director'].fillna('Unknown')
netflix_data['cast'] = netflix_data['cast'].fillna('Unknown')
netflix_data['country'] = netflix_data['country'].fillna('Unknown')
netflix_data = netflix_data.dropna(subset=['date_added'])

In [50]:
netflix_data['genre'] = netflix_data['listed_in'].str.split(', ')

In [51]:
netflix_data['date_added'] = pd.to_datetime(netflix_data['date_added'].str.strip())

In [52]:
netflix_data['year_added'] = netflix_data['date_added'].dt.year.astype(int)

In [53]:
netflix_data.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,genre,year_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",[Documentaries],2021
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...","[International TV Shows, TV Dramas, TV Mysteries]",2021
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,"[Crime TV Shows, International TV Shows, TV Ac...",2021
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...","[Docuseries, Reality TV]",2021
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,"[International TV Shows, Romantic TV Shows, TV...",2021


In [54]:
netflix_filtered = netflix_data[netflix_data['year_added'].between(2016,2021)].copy()
netflix_filtered['era'] = netflix_filtered['year_added'].apply(
    lambda x: 'pre-covid' if x <= 2019 else 'post-covid'
)

## Section 2 - Growth over time
How many titles did Netflix add each year between 2016 to 2021? Did movies or TV shows grow faster?

In [55]:
content_growth = (netflix_filtered.groupby(['year_added','type'])['title']
                  .count()
                  .reset_index()
                  .rename(columns={'title':'count'})
                  )

fig = px.line(content_growth,
              x='year_added',
              y='count',
              color='type',
              title = 'Netflix content growth by type (2016-2021)',
              labels={
                  'year_added':'year',
                  'count':'Number of titles',
                  'type':'content type'
              },
              color_discrete_map={
                   'Movie': '#E50914',
                   'TV Show': '#221F1F'
              }
              )
fig.update_xaxes(tickvals=list(range(2015,2022)))
fig.show()

In [56]:
content_changes = content_growth.groupby(['year_added','type']).sum('count')
content_pivot = content_growth.pivot(index='year_added', columns='type', values='count')

# calculate year over year pct change for each type
content_pivot['movie_pct_change'] = (content_pivot['Movie'].pct_change() * 100).round(1)
content_pivot['tvshow_pct_change'] = (content_pivot['TV Show'].pct_change() * 100).round(1)

print(content_pivot)                                                                                                                   

type        Movie  TV Show  movie_pct_change  tvshow_pct_change
year_added                                                     
2016          253      176               NaN                NaN
2017          839      349             231.6               98.3
2018         1237      412              47.4               18.1
2019         1424      592              15.1               43.7
2020         1284      595              -9.8                0.5
2021          993      505             -22.7              -15.1


### Key insight
Netflix's content library grew significantly year over year, with movies consistently 
outnumbering TV shows across the entire period. Movie additions grew explosively from 
2016 to 2017 (+231.6%), before growth slowed to +47.4% in 2018 and +15.1% in 2019; suggesting Netflix's initial content acquisition phase was slowing down even before COVID hit.

The impact of COVID-19 is clearly visible from 2020 onwards. Movie additions declined 
-9.8% in 2020 and accelerated further to -22.7% in 2021, reflecting the prolonged 
impact of production shutdowns on film content. TV shows proved more resilient, 
remaining virtually flat in 2020 (+0.5%) before declining -15.1% in 2021, likely 
because TV productions are shorter and easier to resume than feature films.

## Section 3 - Genre shift
Which genre dominated before covid (2016 - 2019) vs after (2020-2021)

In [57]:
netflix_exploded = netflix_filtered.explode('genre')
netflix_exploded.groupby(['era','genre'])['title'].count()

era         genre                   
post-covid  Action & Adventure          366
            Anime Features               37
            Anime Series                 86
            British TV Shows             74
            Children & Family Movies    292
                                       ... 
pre-covid   TV Sci-Fi & Fantasy          37
            TV Shows                      9
            TV Thrillers                 28
            Teen TV Shows                31
            Thrillers                   328
Name: title, Length: 84, dtype: int64

In [58]:
top_genres = (netflix_exploded
              .groupby(['era','genre'])['title']
              .count()
              .reset_index()
              .rename(columns = {'title':'count'})
              .sort_values(['era','count'], ascending=[True,False])
              .groupby('era')
              .head(3)
              )
top_genres

,era,genre,count
16,post-covid,International Movies,983
12,post-covid,Dramas,947
7,post-covid,Comedies,715
58,pre-covid,International Movies,1755
54,pre-covid,Dramas,1453
49,pre-covid,Comedies,942


In [59]:
# count titles per era per genre
genre_count = (netflix_exploded
               .groupby(['era','genre'])['title']
               .count()
               .reset_index()
               .rename(columns ={'title':'count'})
               )

# divide by number of years in each era to get average per year 
genre_count['years'] = genre_count['era'].map({
    'pre-covid': 4, 
    'post-covid': 2
})

genre_count['avg_per_year'] = genre_count['count'] / genre_count['years']

top_genres = (genre_count 
              .sort_values(['era','avg_per_year'], ascending=[True,False])
              .groupby('era')
              .head(3)
              )
print(top_genres)

           era                 genre  count  years  avg_per_year
16  post-covid  International Movies    983      2        491.50
12  post-covid                Dramas    947      2        473.50
7   post-covid              Comedies    715      2        357.50
58   pre-covid  International Movies   1755      4        438.75
54   pre-covid                Dramas   1453      4        363.25
49   pre-covid              Comedies    942      4        235.50


In [60]:
pivot_df = top_genres.pivot(index = 'genre', columns = 'era', values ='avg_per_year')
pivot_df['pct_change'] = ((pivot_df['post-covid'] - pivot_df['pre-covid']) / pivot_df['pre-covid'] *100).round(1)
pivot_df

era,post-covid,pre-covid,pct_change
genre,,,
Comedies,357.5,235.50,51.8
Dramas,473.5,363.25,30.4
International Movies,491.5,438.75,12.0


In [78]:
# prepare data
genre_viz = (netflix_filtered
    .explode('genre')
    .groupby(['era', 'genre'])['title']
    .count()
    .reset_index()
    .rename(columns={'title': 'count'})
)

# normalize by years
genre_viz['avg_per_year'] = genre_viz.apply(
    lambda x: x['count'] / 4 if x['era'] == 'pre-covid' else x['count'] / 2, axis=1
).round(1)

# get top 5 overall genres
top_genres = genre_viz.groupby('genre')['avg_per_year'].sum().nlargest(5).index
genre_viz_filtered = genre_viz[genre_viz['genre'].isin(top_genres)]
genre_viz_filtered['genre'] = genre_viz_filtered['genre'].str.replace('International ', 'Intl. ')

fig3 = px.bar(genre_viz_filtered,
    x='genre',
    y='avg_per_year',
    color='era',
    barmode='group',
    title='Top 5 genres: pre vs post COVID (avg per year)',
    labels={
        'genre': 'Genre',
        'avg_per_year': 'Avg titles per year',
        'era': 'Era'
    },
    color_discrete_map={
        'pre-covid': '#221F1F',
        'post-covid': '#E50914'
    }
)
fig3.for_each_trace(lambda t: t.update(name=t.name.replace(
    'post-covid', 'Post-COVID').replace('pre-covid', 'Pre-COVID')
))
fig3.show()

/var/folders/z_/lsl6pgzx1tncsw0zw834vjx00000gn/T/ipykernel_91862/1713264727.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### Key insight
After normalizing for time, all three top genre domination both post- and pre- covid actually **increased** post-covid.
Comedies saw the sharpest growth at ~49% more titles per year, suggesting Netflix leaned into lighter content during the pandemic. Internation movies grew ~11%, supporting the thesis that Netflix accelerated its global content push after 2020.

## Section 4 - International push
Did Netflix invest more in non-US content after the pandemic began?

**Note on methodology:** Analysis is filtered to 2016 onwards. Years prior to 2016 were excluded due to insufficient data. For example, 2008-2014 had fewer than 25 titles per year, making percentage comparisions unreliable and potentially misleading.

In [62]:
netflix_filtered['is_us'] = netflix_filtered['country'].str.contains('United States', na = False)
netflix_filtered.groupby(['year_added', 'is_us'])['title'].count().unstack()

is_us,False,True
year_added,,
2016,226,203
2017,726,462
2018,1049,600
2019,1160,856
2020,1051,828
2021,871,627


In [63]:
internation_growth = (netflix_filtered.groupby(['year_added','is_us'])['title']
                      .count()
                      .unstack()
                      .fillna(0)
                      .astype(int)
                      )
internation_growth.columns = ['International','US']
internation_growth

,International,US
year_added,,
2016,226,203
2017,726,462
2018,1049,600
2019,1160,856
2020,1051,828
2021,871,627


### Initial approach - era averages
Looking at averages per era gives a high level overview, however averaging across multiple years can mask important year-by-year tends.

In [64]:
internation_growth['total'] = internation_growth['International'] + internation_growth['US']
internation_growth['International_pct'] = (
    (internation_growth['International'] / internation_growth['total'] *100)
).round(1)
print(internation_growth)

            International   US  total  International_pct
year_added                                              
2016                  226  203    429               52.7
2017                  726  462   1188               61.1
2018                 1049  600   1649               63.6
2019                 1160  856   2016               57.5
2020                 1051  828   1879               55.9
2021                  871  627   1498               58.1


In [65]:
pre = internation_growth.loc[2016:2019, 'International_pct'].mean().round(1)
post = internation_growth.loc[2020:2021, 'International_pct'].mean().round(1)
change = (post - pre).round(1)

print(f"Pre-COVID avg internal %: {pre}%")
print (f"Post-COVID avg international %: {post}%")
print(f"Change: {change}%")

Pre-COVID avg internal %: 58.7%
Post-COVID avg international %: 57.0%
Change: -1.7%


### Refined approach - year by year breakdown
Breaking this down annually reveals a more nuanced three-act story that averages alone would have hidden.

In [66]:
print(internation_growth.loc[2016:2021, 'International_pct'])

year_added
2016    52.7
2017    61.1
2018    63.6
2019    57.5
2020    55.9
2021    58.1
Name: International_pct, dtype: float64


In [76]:
# prepare data
intl_viz = (netflix_filtered
    .groupby(['year_added', 'is_us'])['title']
    .count()
    .unstack()
    .fillna(0)
    .astype(int)
    .reset_index()
)
intl_viz.columns = ['year_added', 'international', 'us']
intl_viz['total'] = intl_viz['international'] + intl_viz['us']
intl_viz['international_pct'] = (intl_viz['international'] / intl_viz['total'] * 100).round(1)

fig4 = px.line(intl_viz,
    x='year_added',
    y='international_pct',
    title='International content % by year (2016–2021)',
    labels={
        'year_added': 'Year',
        'international_pct': 'International content %'
    },
    markers=True
)

fig4.update_traces(line_color='#E50914', marker_color='#221F1F')

# add reference line at 50%
fig4.add_hline(
    y=50,
    line_dash='dash',
    line_color='gray',
    annotation_text="50% threshold",
    annotation_position="bottom right"
)
fig4.add_annotation(
    x=2018, y=63.6,
    text="Peak: 63.6%",
    showarrow=True,
    arrowhead=2,
    yshift=15,
    font=dict(color='#E50914')
)
fig4.update_xaxes(tickvals=list(range(2016, 2022)))
fig4.show()

### Key insight
International content grew steadily from 52.7% in 2016 to a peak of 63.6% in 2018, suggesting Netflix was aggressively expanding globally during this period. However, this trend revered in 2019 and dipped further in 2020 to 55.9%, likely refelcting worldwide production shutdowns during the pandemic. The slight recovery to 58.1% in 2021 suggest Netflix's international strategy was beginning to bounce back as production resumed. 

## Section 5 - Content maturity 
DId Netflix shift towards more mature content rating post-COVID?

*Note on methodology*: Analysis is filtered to 2016-2021 using `netflix_filtered` to ensure consistency with previous section. Counts are normalized by years (pre-covid = 4 years, post-covid = 2 years) to allow fair comparison.

In [67]:
rating_counts = (netflix_filtered.groupby(['era','rating'])['title']
                 .count()
                 .unstack(level = 0)
                 .fillna(0)
                 .astype(int)
                 )

# normalize by years per era
rating_counts['pre_covid_avg'] = (rating_counts['pre-covid']/4).round(1)
rating_counts['post_covid_avg'] = (rating_counts['post-covid']/2).round(1)

# calculate percent change

rating_counts['pct_change'] = (
    (rating_counts['post_covid_avg'] - rating_counts['pre_covid_avg']) / rating_counts['pre_covid_avg'] *100
).round(1)

rating_counts

# top 3 per era
rating_summary = (rating_counts[['pre_covid_avg','post_covid_avg','pct_change']]
                  .sort_values('post_covid_avg',ascending=False)
                  ).head(5)
print(rating_summary)

era     pre_covid_avg  post_covid_avg  pct_change
rating                                           
TV-MA           498.5           580.0        16.3
TV-14           342.2           382.5        11.8
R               104.2           189.0        81.4
PG-13            55.0           134.0       143.6
TV-PG           150.0           121.5       -19.0


In [77]:
# prepare data
ratings_viz = rating_counts[['pre_covid_avg', 'post_covid_avg']].reset_index()
ratings_viz = ratings_viz.sort_values('post_covid_avg', ascending=False).head(5)

# melt for plotly
ratings_melted = ratings_viz.melt(
    id_vars='rating',
    value_vars=['pre_covid_avg', 'post_covid_avg'],
    var_name='era',
    value_name='avg_per_year'
)

# rename era labels
ratings_melted['era'] = ratings_melted['era'].map({
    'pre_covid_avg': 'Pre-COVID',
    'post_covid_avg': 'Post-COVID'
})

fig5 = px.bar(ratings_melted,
    x='rating',
    y='avg_per_year',
    color='era',
    barmode='group',
    title='Top 5 content ratings: pre vs post COVID (avg per year)',
    labels={
        'rating': 'Rating',
        'avg_per_year': 'Avg titles per year',
        'era': 'Era'
    },
    color_discrete_map={
        'Pre-COVID': '#221F1F',
        'Post-COVID': '#E50914'
    }
)
fig5.add_annotation(
    x='PG-13', y=134,
    text="+143.6%",
    showarrow=True,
    arrowhead=2,
    yshift=15,
    font=dict(color='#E50914')
)
fig5.show()

### Key insight
Netflix's content shifted toward more mature ratings post-COVID. TV-PG content 
declined by 19%, dropping out of the top ratings entirely, suggesting a move away 
from family-friendly content. Meanwhile mature ratings surged; TV-MA grew 16.3%, 
TV-14 grew 11.8%, and most notably R-rated content nearly doubled (+81.4%). However,
the sharpest growth was PG-13, which increased by 143.6%, suggesting 
Netflix aggressively expanded its mid-maturity movie catalog post-COVID.

## Section 6 - Seasonal patterns
Does Netflix follow a content release calender? Did COVID disrupt it?

In [68]:
netflix_filtered['month_added'] = netflix_filtered['date_added'].dt.month

seasonal = (netflix_filtered.groupby(['era','month_added'])['title']
            .count()
            .unstack(level = 0)
            .fillna(0)
)

seasonal['pre_covid_avg'] = (seasonal['pre-covid'] /4).round(1)
seasonal['post_covid_avg'] = (seasonal['post-covid']/2).round(1)

seasonal['pct_change'] = ((seasonal['post_covid_avg'] - seasonal['pre_covid_avg']) / seasonal['pre_covid_avg'] *100).round(1)

seasonal


era,post-covid,pre-covid,pre_covid_avg,post_covid_avg,pct_change
month_added,,,,,
1,337,397,99.2,168.5,69.9
2,223,332,83.0,111.5,34.3
3,249,487,121.8,124.5,2.2
4,365,392,98.0,182.5,86.2
5,289,335,83.8,144.5,72.4
6,363,358,89.5,181.5,102.8
7,403,416,104.0,201.5,93.8
8,307,444,111.0,153.5,38.3
9,351,408,102.0,175.5,72.1


In [69]:
best_month = seasonal.loc[seasonal['pct_change'].idxmax(), ['pre_covid_avg','post_covid_avg','pct_change']]
print("Biggest increase:\n", best_month)

worst_month = seasonal.loc[seasonal['pct_change'].idxmin(), ['pre_covid_avg','post_covid_avg','pct_change']]
print("\nBiggest decrease:\n", worst_month,'\n')

# avg pct change for summer months (Mar-sept) vs Q4 (OCt-Dec)
summer = seasonal.loc[seasonal.index.isin(range(3,10)), 'pct_change'].mean().round(1)
q4 = seasonal.loc[seasonal.index.isin(range(10,13)), 'pct_change'].mean().round(1)
print(f"Average % change Mar-Sep: {summer}%")
print(f"Average % change Oct-Dec: {q4}%")

Biggest increase:
 era
pre_covid_avg      89.5
post_covid_avg    181.5
pct_change        102.8
Name: 6, dtype: float64

Biggest decrease:
 era
pre_covid_avg     153.5
post_covid_avg     84.5
pct_change        -45.0
Name: 12, dtype: float64 

Average % change Mar-Sep: 66.8%
Average % change Oct-Dec: -42.7%


In [70]:
seasonal_melted = seasonal[['pre_covid_avg','post_covid_avg']].reset_index()

fig = px.line(seasonal_melted, 
              x ='month_added', 
              y =['pre_covid_avg','post_covid_avg'],
              title = 'Monthly content additions: pre vs post COVID',
              labels = {'month_added': 'Month',
                        'value': 'Avg title added',
                        'variable': 'Era'
                        }, 
            color_discrete_map= {
                'pre_covid_avg':'#221F1F',
                'post_covid_avg': '#E50914'
            }
        )

newnames = {'pre_covid_avg':'Pre-COVID', 'post_covid_avg': 'Post-COVID'}
fig.for_each_trace(lambda t: t.update(name =newnames[t.name]))
fig.update_xaxes(
    tickvals=list(range(1,13)),
    ticktext=['Jan','Feb','Mar', 'Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
)
fig.show()

In [71]:

pre_dist = seasonal['pre_covid_avg'] / seasonal['pre_covid_avg'].sum()
post_dist = seasonal['post_covid_avg'] / seasonal['post_covid_avg'].sum()
observed_tvd = 0.5 * np.abs(pre_dist - post_dist).sum()

# simulate under null hypothesis
n_permutations = 10000
perm_tvds = []

combined = np.concatenate([seasonal['pre_covid_avg'].values,
                           seasonal['post_covid_avg'].values])

for _ in range(n_permutations):
    shuffled = np.random.permutation(combined)
    perm_pre = shuffled[:12] / shuffled[:12].sum()
    perm_post = shuffled[12:] / shuffled[12:].sum()
    perm_tvds.append(0.5 * np.abs(perm_pre - perm_post).sum())

p_value = np.mean(np.array(perm_tvds) >= observed_tvd)
print(f"Observed TVD: {observed_tvd:.3f}")
print(f"P-value: {p_value:.3f}")

if p_value < 0.05:
    print("Statistically significant!")
else:
    print("Not statistically significant")

Observed TVD: 0.198
P-value: 0.144
Not statistically significant


### Key insight
Post-COVID Netflix dramatically front-loaded its release calendar. Every month from January through September saw increases in average montly additions, with June seeing the larget jump with 93 more titles per year on average, a +102.8% increase. This reversed sharply in Q4, with december seeing the steepest decline at 69 fewer title per year (-45.0%), suggesting COVID production shutdowns disrupted Netflix's traditional end-of-year content push.

This aligns with widely reported COVID-19 production shutdowns in early 2020, which 
halted filming globally and likely caused Netflix to front-load content releases to 
compensate for anticipated Q4 gaps caused by delayed productions.

### Statistical note
A TVD + permutation test (n=10,000) returned a TVD of 0.198 (p=0.143), falling short 
of the conventional 0.05 threshold. However given that Netflix's release calendar 
reflects deliberate business decisions rather than random processes, the assumption of 
randomness underlying this test may not be appropriate here. The consistent pattern of 
post-COVID increases from January through September and consistent Q4 declines suggest 
a meaningful strategic shift that future analysis with additional years of data could 
confirm more rigorously.

### Limitation
This analysis is based on 6 years of data (2016–2021) with only 2 post-COVID years. 
A more robust analysis would incorporate 2022–2024 data to confirm whether the 
calendar shift persisted or was a temporary COVID response.

## Section 7 - Summary & conclusions

Key findings from the analysis of Netflix's content strategy evolution from 2016 to 2021 through the lens of COVID-19.

### Key findings

- **Content growth:** Movie additions grew explosively from 2016 to 2017 (+231.6%) 
  before slowing down to +15.1% by 2019 — suggesting Netflix's content acquisition 
  phase was already slowing before COVID. Movies declined -9.8% in 2020 and -22.7% 
  in 2021, while TV shows proved more resilient, remaining virtually flat in 2020 
  (+0.5%) before declining -15.1% in 2021.

- **Genre shifts:** All top 3 genres increased post-COVID when normalized by year. 
  Comedies saw the sharpest growth at +49.1%, suggesting Netflix leaned into lighter 
  content during the pandemic period.

- **International content:** International content peaked at 63.6% in 2018 before 
  dipping to 55.9% in 2020, suggesting COVID disrupted rather than accelerated 
  Netflix's global push. A partial recovery to 58.1% in 2021 suggests the 
  international strategy was beginning to bounce back.

- **Content maturity:** Netflix shifted toward more mature content post-COVID; 
  PG-13 grew the most sharply (+143.6%), R-rated content nearly doubled (+81.4%), 
  while family-friendly TV-PG declined -19.0%.

- **Seasonal patterns:** Netflix dramatically front-loaded its release calendar 
  post-COVID, with June peaking at +102.8% more titles per year. This reversed 
  sharply in Q4 with December declining -45.0%, consistent with global production 
  shutdowns disrupting Netflix's traditional end-of-year content push.


### Overall conclusion
Taken together, the data suggests COVID-19 acted as a catalyst that accelerated 
and in some cases disrupted existing Netflix trends rather than creating entirely 
new ones. Content growth was already slowing before 2020, international expansion 
had peaked in 2018, and the shift toward mature content likely reflects both 
changing viewer preferences during lockdowns and practical production constraints. 
TV shows proved more resilient than movies to COVID disruption, possibly due to 
shorter production cycles.

### Limitations
- Dataset covers 2016–2021 only; post-COVID window limited to 2 years
- 9% of titles missing country data excluded from international analysis
- Statistical significance limited by small sample size (TVD p=0.143)
- No data on viewership or subscriber growth to validate content strategy impact

### Future work
- Incorporate 2022–2024 data to confirm whether trends persisted post-pandemic
- Analyze Netflix original vs licensed content separately
- Cross-reference with Netflix subscriber growth data for the same period
- Apply a recommender system to predict what content Netflix might prioritize next